In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Load silver stocks data
print("📊 Loading silver stocks data...")
silver_stocks = spark.table("financial_market.silver.stocks")

print(f"Stocks records: {silver_stocks.count():,}")
print(f"Date range: {silver_stocks.agg(F.min('date')).collect()[0][0]} to {silver_stocks.agg(F.max('date')).collect()[0][0]}")
print(f"Unique tickers: {silver_stocks.select('ticker').distinct().count()}")

# Load silver crypto data
print("\n💰 Loading silver crypto data...")
silver_crypto = spark.table("financial_market.silver.crypto")

print(f"Crypto records: {silver_crypto.count():,}")
print(f"Date range: {silver_crypto.agg(F.min('date')).collect()[0][0]} to {silver_crypto.agg(F.max('date')).collect()[0][0]}")
print(f"Unique cryptos: {silver_crypto.select('crypto_id').distinct().count()}")

print("\n✅ Data loaded successfully!")

In [0]:
# ============================
# STOCK Z-SCORE CALCULATION
# Rolling 30-day window — production ready
# ============================
print("📊 Calculating rolling z-scores for stocks...\n")

# Rolling window — 30 days partitioned by ticker
window_30d = Window \
    .partitionBy("ticker") \
    .orderBy("date") \
    .rowsBetween(-29, 0)

# Rolling mean + stddev
stocks_zscore = silver_stocks \
    .withColumn("rolling_mean_change",
                F.avg("daily_change_pct").over(window_30d)) \
    .withColumn("rolling_std_change",
                F.stddev("daily_change_pct").over(window_30d)) \
    .withColumn("rolling_mean_volume",
                F.avg("volume").over(window_30d)) \
    .withColumn("rolling_std_volume",
                F.stddev("volume").over(window_30d)) \
    .withColumn("rolling_mean_volatility",
                F.avg("price_volatility").over(window_30d)) \
    .withColumn("rolling_std_volatility",
                F.stddev("price_volatility").over(window_30d))

# Z-score calculate karo
stocks_zscore = stocks_zscore \
    .withColumn("zscore_daily_change",
                F.when(F.col("rolling_std_change") > 0,
                    (F.col("daily_change_pct") - F.col("rolling_mean_change"))
                    / F.col("rolling_std_change"))
                .otherwise(0)) \
    .withColumn("zscore_volume",
                F.when(F.col("rolling_std_volume") > 0,
                    (F.col("volume") - F.col("rolling_mean_volume"))
                    / F.col("rolling_std_volume"))
                .otherwise(0)) \
    .withColumn("zscore_volatility",
                F.when(F.col("rolling_std_volatility") > 0,
                    (F.col("price_volatility") - F.col("rolling_mean_volatility"))
                    / F.col("rolling_std_volatility"))
                .otherwise(0))

# Round karo
stocks_zscore = stocks_zscore \
    .withColumn("zscore_daily_change",
                F.round(F.col("zscore_daily_change"), 2)) \
    .withColumn("zscore_volume",
                F.round(F.col("zscore_volume"), 2)) \
    .withColumn("zscore_volatility",
                F.round(F.col("zscore_volatility"), 2))

# Anomaly flags
stocks_zscore = stocks_zscore \
    .withColumn("is_price_anomaly",
                F.abs(F.col("zscore_daily_change")) > 2) \
    .withColumn("is_volume_anomaly",
                F.col("zscore_volume") > 2) \
    .withColumn("is_volatility_anomaly",
                F.col("zscore_volatility") > 2) \
    .withColumn("anomaly_count",
                F.when(F.col("is_price_anomaly"), 1).otherwise(0) +
                F.when(F.col("is_volume_anomaly"), 1).otherwise(0) +
                F.when(F.col("is_volatility_anomaly"), 1).otherwise(0))

print(f"✅ Total stock records: {stocks_zscore.count():,}")
print(f"✅ Price anomalies: {stocks_zscore.filter(F.col('is_price_anomaly')).count():,}")
print(f"✅ Volume anomalies: {stocks_zscore.filter(F.col('is_volume_anomaly')).count():,}")
print(f"✅ Volatility anomalies: {stocks_zscore.filter(F.col('is_volatility_anomaly')).count():,}")
print(f"✅ Records with any anomaly: {stocks_zscore.filter(F.col('anomaly_count') > 0).count():,}")

print("\n📋 Top Stock Anomalies:")
display(stocks_zscore
    .filter(F.col("anomaly_count") > 0)
    .select("ticker", "date", "close", "daily_change_pct",
            "zscore_daily_change", "zscore_volume", "zscore_volatility",
            "is_price_anomaly", "is_volume_anomaly",
            "is_volatility_anomaly", "anomaly_count")
    .orderBy(F.col("date").desc(), F.col("anomaly_count").desc())
    .limit(10))

In [0]:
# ============================
# CRYPTO Z-SCORE CALCULATION
# Rolling 30-day window — production ready
# ============================
print("💰 Calculating rolling z-scores for crypto...\n")

# Calculate price change percentage from historical prices
lag_window = Window.partitionBy("crypto_id").orderBy("date")
crypto_with_change = silver_crypto \
    .withColumn("prev_price", F.lag("price_usd", 1).over(lag_window)) \
    .withColumn("price_change_pct",
                F.when(F.col("prev_price").isNotNull() & (F.col("prev_price") > 0),
                    ((F.col("price_usd") - F.col("prev_price")) / F.col("prev_price")) * 100)
                .otherwise(0))

# Rolling window — 30 days partitioned by crypto_id
crypto_window_30d = Window \
    .partitionBy("crypto_id") \
    .orderBy("date") \
    .rowsBetween(-29, 0)

# Rolling mean + stddev
crypto_zscore = crypto_with_change \
    .withColumn("rolling_mean_price_change",
                F.avg("price_change_pct").over(crypto_window_30d)) \
    .withColumn("rolling_std_price_change",
                F.stddev("price_change_pct").over(crypto_window_30d)) \
    .withColumn("rolling_mean_volume",
                F.avg("volume_24h_millions").over(crypto_window_30d)) \
    .withColumn("rolling_std_volume",
                F.stddev("volume_24h_millions").over(crypto_window_30d)) \
    .withColumn("rolling_mean_market_cap",
                F.avg("market_cap_billions").over(crypto_window_30d)) \
    .withColumn("rolling_std_market_cap",
                F.stddev("market_cap_billions").over(crypto_window_30d))

# Z-score calculate karo
crypto_zscore = crypto_zscore \
    .withColumn("zscore_price_change",
                F.when(F.col("rolling_std_price_change") > 0,
                    (F.col("price_change_pct") - F.col("rolling_mean_price_change"))
                    / F.col("rolling_std_price_change"))
                .otherwise(0)) \
    .withColumn("zscore_volume",
                F.when(F.col("rolling_std_volume") > 0,
                    (F.col("volume_24h_millions") - F.col("rolling_mean_volume"))
                    / F.col("rolling_std_volume"))
                .otherwise(0)) \
    .withColumn("zscore_market_cap",
                F.when(F.col("rolling_std_market_cap") > 0,
                    (F.col("market_cap_billions") - F.col("rolling_mean_market_cap"))
                    / F.col("rolling_std_market_cap"))
                .otherwise(0))

# Round karo
crypto_zscore = crypto_zscore \
    .withColumn("zscore_price_change",
                F.round(F.col("zscore_price_change"), 2)) \
    .withColumn("zscore_volume",
                F.round(F.col("zscore_volume"), 2)) \
    .withColumn("zscore_market_cap",
                F.round(F.col("zscore_market_cap"), 2))

# Anomaly flags
crypto_zscore = crypto_zscore \
    .withColumn("is_price_anomaly",
                F.abs(F.col("zscore_price_change")) > 2) \
    .withColumn("is_volume_anomaly",
                F.col("zscore_volume") > 2) \
    .withColumn("is_market_cap_anomaly",
                F.abs(F.col("zscore_market_cap")) > 2) \
    .withColumn("anomaly_count",
                F.when(F.col("is_price_anomaly"), 1).otherwise(0) +
                F.when(F.col("is_volume_anomaly"), 1).otherwise(0) +
                F.when(F.col("is_market_cap_anomaly"), 1).otherwise(0))

print(f"✅ Total crypto records: {crypto_zscore.count():,}")
print(f"✅ Price anomalies: {crypto_zscore.filter(F.col('is_price_anomaly')).count():,}")
print(f"✅ Volume anomalies: {crypto_zscore.filter(F.col('is_volume_anomaly')).count():,}")
print(f"✅ Market cap anomalies: {crypto_zscore.filter(F.col('is_market_cap_anomaly')).count():,}")
print(f"✅ Records with any anomaly: {crypto_zscore.filter(F.col('anomaly_count') > 0).count():,}")

print("\n📋 Crypto Anomalies:")
display(crypto_zscore
    .filter(F.col("anomaly_count") > 0)
    .select("crypto_id", "date", "price_usd",
            "price_change_pct", "volume_24h_millions",
            "zscore_price_change", "zscore_volume", "zscore_market_cap",
            "is_price_anomaly", "is_volume_anomaly",
            "is_market_cap_anomaly", "anomaly_count")
    .orderBy(F.col("date").desc(), F.col("anomaly_count").desc())
    .limit(10))

In [0]:
# ============================
# ANOMALY SUMMARY & SAVE
# ============================

print("📊 Generating anomaly summary...\n")

# Create stock anomaly summary
stock_anomaly_summary = stocks_zscore \
    .filter(F.col("anomaly_count") > 0) \
    .groupBy("ticker").agg(
        F.count("*").alias("total_anomalies"),
        F.sum(F.when(F.col("is_price_anomaly"), 1).otherwise(0)).alias("price_anomalies"),
        F.sum(F.when(F.col("is_volume_anomaly"), 1).otherwise(0)).alias("volume_anomalies"),
        F.sum(F.when(F.col("is_volatility_anomaly"), 1).otherwise(0)).alias("volatility_anomalies"),
        F.max(F.abs(F.col("zscore_daily_change"))).alias("max_price_zscore"),
        F.max(F.col("zscore_volume")).alias("max_volume_zscore"),
        F.max(F.col("zscore_volatility")).alias("max_volatility_zscore"),
        F.min("date").alias("first_anomaly_date"),
        F.max("date").alias("last_anomaly_date")
    ).orderBy(F.col("total_anomalies").desc())

print("Stock Anomaly Summary:")
print(f"  Tickers with anomalies: {stock_anomaly_summary.count()}")
display(stock_anomaly_summary)

# Create crypto anomaly summary
crypto_anomaly_summary = crypto_zscore \
    .filter(F.col("anomaly_count") > 0) \
    .groupBy("crypto_id").agg(
        F.count("*").alias("total_anomalies"),
        F.sum(F.when(F.col("is_price_anomaly"), 1).otherwise(0)).alias("price_anomalies"),
        F.sum(F.when(F.col("is_volume_anomaly"), 1).otherwise(0)).alias("volume_anomalies"),
        F.sum(F.when(F.col("is_market_cap_anomaly"), 1).otherwise(0)).alias("market_cap_anomalies"),
        F.max(F.abs(F.col("zscore_price_change"))).alias("max_price_zscore"),
        F.max(F.col("zscore_volume")).alias("max_volume_zscore"),
        F.max(F.abs(F.col("zscore_market_cap"))).alias("max_market_cap_zscore"),
        F.min("date").alias("first_anomaly_date"),
        F.max("date").alias("last_anomaly_date")
    ).orderBy(F.col("total_anomalies").desc())

print("\nCrypto Anomaly Summary:")
print(f"  Symbols with anomalies: {crypto_anomaly_summary .count()}")
display(crypto_anomaly_summary)

# Save to gold layer
print("\n📦 Saving anomaly data to gold layer...")

# Ensure gold schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS financial_market.gold")

# Select relevant columns for gold layer
stocks_anomalies_gold = stocks_zscore.select(
    "ticker", "date", "open", "high", "low", "close", "volume",
    "daily_change_pct", "price_volatility",
    "zscore_daily_change", "zscore_volume", "zscore_volatility",
    "is_price_anomaly", "is_volume_anomaly", "is_volatility_anomaly", "anomaly_count"
).withColumn("processed_timestamp", F.current_timestamp())

crypto_anomalies_gold = crypto_zscore.select(
    "crypto_id", "date", "price_usd", "market_cap_billions", "volume_24h_millions",
    "price_change_pct",
    "zscore_price_change", "zscore_volume", "zscore_market_cap",
    "is_price_anomaly", "is_volume_anomaly", "is_market_cap_anomaly", "anomaly_count"
).withColumn("processed_timestamp", F.current_timestamp())

# Save stocks anomalies
stocks_anomalies_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.stocks_anomalies")
print("✅ Saved to financial_market.gold.stocks_anomalies")

# Save crypto anomalies
crypto_anomalies_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.crypto_anomalies")
print("✅ Saved to financial_market.gold.crypto_anomalies")

# Save summary tables
stock_anomaly_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.stocks_anomaly_summary")
print("✅ Saved to financial_market.gold.stocks_anomaly_summary")

crypto_anomaly_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.crypto_anomaly_summary")
print("✅ Saved to financial_market.gold.crypto_anomaly_summary")

print("\n✅ All anomaly tables saved successfully!")

In [0]:
# ============================
# CRITICAL ANOMALIES & ALERTS
# ============================

print("🚨 Identifying critical anomalies (|z-score| > 3)...\n")

# Severe stock anomalies
severe_stock_anomalies = stocks_zscore.filter(
    (F.abs(F.col("zscore_daily_change")) > 3) |
    (F.col("zscore_volume") > 3) |
    (F.col("zscore_volatility") > 3)
).select(
    "ticker", "date", "close", "daily_change_pct", "volume", "price_volatility",
    "zscore_daily_change", "zscore_volume", "zscore_volatility"
).orderBy(F.col("date").desc())

print(f"📊 SEVERE STOCK ANOMALIES (z-score > 3): {severe_stock_anomalies.count()}")
if severe_stock_anomalies.count() > 0:
    display(severe_stock_anomalies.limit(10))
else:
    print("  No severe stock anomalies detected.\n")

# Severe crypto anomalies
severe_crypto_anomalies = crypto_zscore.filter(
    (F.abs(F.col("zscore_price_change")) > 3) |
    (F.col("zscore_volume") > 3) |
    (F.abs(F.col("zscore_market_cap")) > 3)
).select(
    "crypto_id", "date", "price_usd", "price_change_pct", 
    "volume_24h_millions", "market_cap_billions",
    "zscore_price_change", "zscore_volume", "zscore_market_cap"
).orderBy(F.col("date").desc())

print(f"\n💰 SEVERE CRYPTO ANOMALIES (z-score > 3): {severe_crypto_anomalies.count()}")
if severe_crypto_anomalies.count() > 0:
    display(severe_crypto_anomalies.limit(10))
else:
    print("  No severe crypto anomalies detected.\n")

# Recent multi-anomaly events (multiple anomaly types simultaneously)
print("\n🔥 MULTI-ANOMALY EVENTS (Recent):")
multi_anomaly_stocks = stocks_zscore.filter(F.col("anomaly_count") >= 2).select(
    "ticker", "date", "close", "daily_change_pct", "volume",
    "zscore_daily_change", "zscore_volume", "zscore_volatility", "anomaly_count"
).orderBy(F.col("date").desc(), F.col("anomaly_count").desc())

print(f"Stocks with 2+ anomaly types: {multi_anomaly_stocks.count()}")
if multi_anomaly_stocks.count() > 0:
    display(multi_anomaly_stocks.limit(5))

multi_anomaly_crypto = crypto_zscore.filter(F.col("anomaly_count") >= 2).select(
    "crypto_id", "date", "price_usd", "price_change_pct",
    "zscore_price_change", "zscore_volume", "zscore_market_cap", "anomaly_count"
).orderBy(F.col("date").desc(), F.col("anomaly_count").desc())

print(f"\nCrypto with 2+ anomaly types: {multi_anomaly_crypto.count()}")
if multi_anomaly_crypto.count() > 0:
    display(multi_anomaly_crypto.limit(5))

print("\n✅ Anomaly detection complete!")
print("\n📊 Summary:")
print(f"  - Total stock anomalies: {stocks_zscore.filter(F.col('anomaly_count') > 0).count():,}")
print(f"  - Total crypto anomalies: {crypto_zscore.filter(F.col('anomaly_count') > 0).count():,}")
print(f"  - Severe stock anomalies: {severe_stock_anomalies.count():,}")
print(f"  - Severe crypto anomalies: {severe_crypto_anomalies.count():,}")
print("\n📦 Gold Layer Tables:")
print("  - financial_market.gold.stocks_anomalies")
print("  - financial_market.gold.crypto_anomalies")
print("  - financial_market.gold.stocks_anomaly_summary")
print("  - financial_market.gold.crypto_anomaly_summary")